# 1. 모듈 import, 드라이브 연결 및 데이터 load

In [1]:
import os
from google.colab import drive

import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from torchvision import transforms

import tqdm
from sklearn.model_selection import train_test_split


### 드라이브 연결
 1. Goodle Drive 접속
 2. 공유 문서함 에서 Synapse 폴더를 찾습니다.
 3. 폴더 우클릭 → [드라이브에 바로가기 추가] 선택.
 4. [내 드라이브]를 위치로 지정하고 추가합니다.


In [9]:
drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/Synapse'
train_csv_path = '/content/drive/MyDrive/Synapse/dataset/train.csv'
test_csv_path =  '/content/drive/MyDrive/Synapse/dataset/test.csv'

train_dir =  '/content/drive/MyDrive/Synapse/dataset/train'
test_dir =  '/content/drive/MyDrive/Synapse/dataset/test'

sample_submission_path =  '/content/drive/MyDrive/Synapse/dataset/sample_submission.csv'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#### 데이터 확인

In [26]:
df = pd.read_csv(train_csv_path)

# # 하위 5개 행 조회
# print(df.tail()

# # 상위 5개 행 조회
print("\n----------상위5개행-----------")
print(df.head())

# 데이터 정보 확인
print("\n----------info-----------")
print(df.info())


print("\n--------class별 분포--------")
print(df['label'].value_counts().sort_index())
test_data = pd.read_csv(test_csv_path)

print("\n--------test data--------")
print(test_data.head())
print(test_data.info())


----------상위5개행-----------
  file_name  label
0   001.PNG      9
1   002.PNG      4
2   003.PNG      1
3   004.PNG      1
4   005.PNG      6

----------info-----------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 723 entries, 0 to 722
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   file_name  723 non-null    object
 1   label      723 non-null    int64 
dtypes: int64(1), object(1)
memory usage: 11.4+ KB
None

--------class별 분포--------
label
0    67
1    71
2    75
3    71
4    67
5    75
6    75
7    72
8    74
9    76
Name: count, dtype: int64

--------test data--------
  file_name
0   001.PNG
1   002.PNG
2   003.PNG
3   004.PNG
4   005.PNG
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199 entries, 0 to 198
Data columns (total 1 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   file_name  199 non-null    object
dtypes: object(1)
memory usage: 1.7+ KB
None


### 난수 시드 고정


In [27]:
import random
import numpy as np

def set_seed(seed=5):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(5)

### train / validation 분리, dataset/dataloader







In [28]:
train_df = pd.read_csv(train_csv_path)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df['label'],
    random_state=42
)

print(len(train_df))
print(len(val_df))

578
145


In [29]:
class CustomDataset(Dataset):

    def __init__(
        self,
        dataframe,
        img_dir,
        transform=None,
        is_test=False
    ):

        self.df = dataframe.reset_index(drop=True)

        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test

        self.file_names = self.df.iloc[:, 0].values

        if not self.is_test:
            self.labels = self.df.iloc[:, 1].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        img_name = self.file_names[idx]

        img_path = os.path.join(
            self.img_dir,
            img_name
        )

        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        if self.is_test:
            return image

        label = self.labels[idx]

        return image, label

In [30]:
# 이미지 형식 확인
import cv2
image_path = '/content/drive/MyDrive/Synapse/dataset/train/001.PNG'
img = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)
height, width, channels = img.shape
print(img.shape)
print(f"이미지 크기: 너비={width}, 높이={height}, 채널={channels}")

(540, 960, 3)
이미지 크기: 너비=960, 높이=540, 채널=3


In [31]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# data set
train_dataset = CustomDataset(train_df, train_dir, transform=train_transform,is_test=False )
val_dataset = CustomDataset(val_df, train_dir, transform=train_transform,is_test=False)

test_df = pd.read_csv(test_csv_path)
test_dataset = CustomDataset(test_df, test_dir, transform=train_transform, is_test=True)

#data Loader
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)

# 2. 모델 정의

### gpu 사용 설정
https://amnesia.tistory.com/72

In [37]:
import torch

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
device = "cuda"

CUDA: True
GPU: Tesla T4


### basic block
cnn에서 사용하는 블록들을 정의합니다
클래스 정의 및 호출 개념은
[링크](https://didu-story.tistory.com/84) 참고해주세요

In [33]:
import torch
import torch.nn as nn

class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, hidden_dim):
        super().__init__()

        self.Conv1 = nn.Conv2d(in_channels, hidden_dim, kernel_size=3, stride=1, padding=1)
        self.Conv2 = nn.Conv2d(hidden_dim, out_channels, kernel_size=3, stride=1, padding=1)
        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):

        x = self.Conv1(x)
        x = self.relu(x)
        x = self.Conv2(x)
        x = self.relu(x)
        x = self.maxpool(x)

        return x

### CNN 정의
위에 정의한 BasicBlock class를 이용하여 CNN 모델을 정의합니다

In [34]:
class CNN(nn.Module):

    def __init__(self, num_classes):

        super(CNN, self).__init__()

        self.block1 = BasicBlock(3, 32, 16)
        self.block2 = BasicBlock(32, 128, 64)
        self.block3 = BasicBlock(128, 256, 128)

        # feature map 크기 고정
        self.gap = nn.AdaptiveAvgPool2d((4, 4))

        # 256 x 4 x 4 = 4096
        self.fc1 = nn.Linear(4096, 2048)
        self.fc2 = nn.Linear(2048, 256)
        self.fc3 = nn.Linear(256, num_classes)

        self.relu = nn.ReLU()

    def forward(self, x):

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)

        # adaptive pooling
        x = self.gap(x)

        # flatten
        x = torch.flatten(x, start_dim=1)

        # classifier
        x = self.fc1(x)
        x = self.relu(x)

        x = self.fc2(x)
        x = self.relu(x)

        x = self.fc3(x)

        return x

### validation 함수

In [35]:
def validate_model(
    model,
    val_loader,
    criterion,
    device
):

    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            total_loss += loss.item() * labels.size(0)

            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()

            total += labels.size(0)

    val_loss = total_loss / total
    val_acc = correct / total

    return val_loss, val_acc

### 학습 실행 함수 정의

In [36]:
def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    device="cuda",
    epochs=10,
    criterion=None,
    scheduler=None,
    save_path=None,
):

    device = torch.device(device)

    model = model.to(device)

    criterion = criterion or nn.CrossEntropyLoss()

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
    }

    best_val_acc = 0.0

    for epoch in range(epochs):

        # =========================
        # Train
        # =========================

        model.train()

        train_loss = 0.0

        train_correct = 0
        train_total = 0

        progress_bar = tqdm.tqdm(
            train_loader,
            desc=f"Epoch [{epoch + 1}/{epochs}]"
        )

        for images, labels in progress_bar:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            batch_size = labels.size(0)

            train_loss += loss.item() * batch_size

            predictions = outputs.argmax(dim=1)

            train_correct += (
                predictions == labels
            ).sum().item()

            train_total += batch_size

            avg_loss = train_loss / train_total
            avg_acc = train_correct / train_total

            progress_bar.set_postfix(
                train_loss=f"{avg_loss:.4f}",
                train_acc=f"{avg_acc:.4f}"
            )

        train_loss /= train_total
        train_acc = train_correct / train_total


        # =========================
        # Validation
        # =========================

        model.eval()

        val_loss = 0.0

        val_correct = 0
        val_total = 0

        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)

                loss = criterion(outputs, labels)

                batch_size = labels.size(0)

                val_loss += loss.item() * batch_size

                predictions = outputs.argmax(dim=1)

                val_correct += (
                    predictions == labels
                ).sum().item()

                val_total += batch_size

        val_loss /= val_total
        val_acc = val_correct / val_total

        # =========================
        # 기록
        # =========================

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)

        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"Epoch [{epoch + 1}/{epochs}] "
            f"| Train Loss: {train_loss:.4f} "
            f"| Train Acc: {train_acc:.4f} "
            f"| Val Loss: {val_loss:.4f} "
            f"| Val Acc: {val_acc:.4f}"
        )

        # scheduler
        if scheduler is not None:
            scheduler.step()

        # best model 저장
        if val_acc > best_val_acc:

            best_val_acc = val_acc

            if save_path:

                torch.save(
                    model.state_dict(),
                    save_path
                )

    print(
        f"Best Validation Accuracy: "
        f"{best_val_acc:.4f}"
    )

    return history

### 학습 실행

In [ ]:
set_seed(5)
device = "cuda"

model = CNN(num_classes=10)


optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    epochs=15,
    save_path=base_path + "/cnn.pt"
)

best_val_acc = max(history["val_acc"])

print(
    f"Best Validation Accuracy: "
    f"{best_val_acc:.4f}"
)

Epoch [1/15]: 100%|██████████| 37/37 [00:13<00:00,  2.67it/s, train_acc=0.0744, train_loss=2.3077]


Epoch [1/15] | Train Loss: 2.3077 | Train Acc: 0.0744 | Val Loss: 2.3033 | Val Acc: 0.0966


Epoch [2/15]: 100%|██████████| 37/37 [00:15<00:00,  2.45it/s, train_acc=0.0952, train_loss=2.3041]


Epoch [2/15] | Train Loss: 2.3041 | Train Acc: 0.0952 | Val Loss: 2.3019 | Val Acc: 0.0966


Epoch [3/15]: 100%|██████████| 37/37 [00:13<00:00,  2.68it/s, train_acc=0.1315, train_loss=2.2199]


...

Epoch [13/15] | Train Loss: 0.5814 | Train Acc: 0.8183 | Val Loss: 1.0439 | Val Acc: 0.6690


Epoch [14/15]: 100%|██████████| 37/37 [00:13<00:00,  2.66it/s, train_acc=0.8512, train_loss=0.4177]


Epoch [14/15] | Train Loss: 0.4177 | Train Acc: 0.8512 | Val Loss: 0.8114 | Val Acc: 0.7448


Epoch [15/15]: 100%|██████████| 37/37 [00:13<00:00,  2.71it/s, train_acc=0.9014, train_loss=0.2469]


Epoch [15/15] | Train Loss: 0.2469 | Train Acc: 0.9014 | Val Loss: 1.0223 | Val Acc: 0.7379
Best Validation Accuracy: 0.7517
Best Validation Accuracy: 0.7517


### 하이퍼파라미터 튜닝

In [ ]:
results = []

learning_rates = [1e-3, 1e-4]
batch_sizes = [16, 32]
optimizers = ["Adam", "AdamW"]

for lr in learning_rates:

    for batch_size in batch_sizes:

        for opt_name in optimizers:
            set_seed(5)
            print("=" * 60)

            print(
                f"LR={lr} | "
                f"Batch={batch_size} | "
                f"Optimizer={opt_name}"
            )

            # -------------------------
            # DataLoader
            # -------------------------

            train_loader = DataLoader(
                train_dataset,
                batch_size=batch_size,
                shuffle=True,
                num_workers=2
            )

            val_loader = DataLoader(
                val_dataset,
                batch_size=batch_size,
                shuffle=False,
                num_workers=2
            )

            # -------------------------
            # Model
            # -------------------------

            model = CNN(
                num_classes=10
            ).to(device)

            # -------------------------
            # Optimizer
            # -------------------------

            if opt_name == "Adam":

                optimizer = torch.optim.Adam(
                    model.parameters(),
                    lr=lr
                )

            else:

                optimizer = torch.optim.AdamW(
                    model.parameters(),
                    lr=lr
                )

            # -------------------------
            # Train
            # -------------------------

            history = train_model(
                model=model,
                train_loader=train_loader,
                val_loader=val_loader,
                optimizer=optimizer,
                device=device,
                epochs=15
            )

            # -------------------------
            # 최고 validation accuracy
            # -------------------------

            best_val_acc = max(
                history["val_acc"]
            )

            # -------------------------
            # 결과 저장
            # -------------------------

            results.append({
                "lr": lr,
                "batch_size": batch_size,
                "optimizer": opt_name,
                "best_val_acc": best_val_acc
            })

LR=0.001 | Batch=16 | Optimizer=Adam


Epoch [1/15]: 100%|██████████| 37/37 [00:13<00:00,  2.65it/s, train_acc=0.0744, train_loss=2.3077]


Epoch [1/15] | Train Loss: 2.3077 | Train Acc: 0.0744 | Val Loss: 2.3033 | Val Acc: 0.0966


Epoch [2/15]: 100%|██████████| 37/37 [00:14<00:00,  2.63it/s, train_acc=0.0952, train_loss=2.3041]


Epoch [2/15] | Train Loss: 2.3041 | Train Acc: 0.0952 | Val Loss: 2.3019 | Val Acc: 0.0966


...

Epoch [13/15] | Train Loss: 0.5254 | Train Acc: 0.8080 | Val Loss: 1.0534 | Val Acc: 0.6621


Epoch [14/15]: 100%|██████████| 19/19 [00:14<00:00,  1.35it/s, train_acc=0.7647, train_loss=0.6781]


Epoch [14/15] | Train Loss: 0.6781 | Train Acc: 0.7647 | Val Loss: 0.8625 | Val Acc: 0.7103


Epoch [15/15]: 100%|██████████| 19/19 [00:13<00:00,  1.40it/s, train_acc=0.8304, train_loss=0.4860]


Epoch [15/15] | Train Loss: 0.4860 | Train Acc: 0.8304 | Val Loss: 0.7172 | Val Acc: 0.7931
Best Validation Accuracy: 0.7931


### 결과 비교

In [ ]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="best_val_acc",
    ascending=False
)

print(results_df)

       lr  batch_size optimizer  best_val_acc
4  0.0001          16      Adam      0.820690
1  0.0010          16     AdamW      0.813793
6  0.0001          32      Adam      0.800000
3  0.0010          32     AdamW      0.793103
7  0.0001          32     AdamW      0.793103
5  0.0001          16     AdamW      0.786207
2  0.0010          32      Adam      0.779310
0  0.0010          16      Adam      0.751724


### test prediction + submission 생성

In [38]:
def predict_gen_submission(csv_name, model, test_loader):
    sample_submission = pd.read_csv(sample_submission_path)
    predictions = []
    model.eval()

    with torch.no_grad():
        for images in test_loader:
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            predictions.extend(preds.cpu().numpy())

    sample_submission.iloc[:, 1] = predictions
    sample_submission.to_csv(base_path + '/' + csv_name + '.csv', index=False)
    print(f'{csv_name}.csv saved')

In [43]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = CNN(num_classes=10)
model.load_state_dict(torch.load(os.path.join(base_path, 'cnn.pt')))
model = model.to(device)

predict_gen_submission(csv_name='submission_cnn', model=model, test_loader=test_loader)

submission_cnn.csv saved


# Agumentation 실험

In [ ]:
set_seed(5)

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# data set
train_dataset = CustomDataset(train_df, train_dir, transform=train_transform,is_test=False )
val_dataset = CustomDataset(val_df, train_dir, transform=train_transform,is_test=False)

test_df = pd.read_csv(test_csv_path)
test_dataset = CustomDataset(test_df, test_dir, transform=train_transform, is_test=True)

#data Loader
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)
device = "cuda"

model = CNN(num_classes=10)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    epochs=30,
    save_path=base_path + "/cnn_epoch30.pt"
)

best_val_acc = max(history["val_acc"])

print(
    f"Best Validation Accuracy: "
    f"{best_val_acc:.4f}"
)

Epoch [1/30]: 100%|██████████| 37/37 [00:14<00:00,  2.59it/s, train_acc=0.0744, train_loss=2.3077]


Epoch [1/30] | Train Loss: 2.3077 | Train Acc: 0.0744 | Val Loss: 2.3033 | Val Acc: 0.0966


Epoch [2/30]: 100%|██████████| 37/37 [00:15<00:00,  2.43it/s, train_acc=0.0952, train_loss=2.3041]


Epoch [2/30] | Train Loss: 2.3041 | Train Acc: 0.0952 | Val Loss: 2.3019 | Val Acc: 0.0966


Epoch [3/30]: 100%|██████████| 37/37 [00:14<00:00,  2.63it/s, train_acc=0.1315, train_loss=2.2199]


...

Epoch [28/30] | Train Loss: 0.1670 | Train Acc: 0.9567 | Val Loss: 0.8208 | Val Acc: 0.7793


Epoch [29/30]: 100%|██████████| 37/37 [00:13<00:00,  2.79it/s, train_acc=0.9585, train_loss=0.1481]


Epoch [29/30] | Train Loss: 0.1481 | Train Acc: 0.9585 | Val Loss: 1.1293 | Val Acc: 0.7379


Epoch [30/30]: 100%|██████████| 37/37 [00:13<00:00,  2.71it/s, train_acc=0.9896, train_loss=0.0402]


Epoch [30/30] | Train Loss: 0.0402 | Train Acc: 0.9896 | Val Loss: 0.8514 | Val Acc: 0.8414
Best Validation Accuracy: 0.8414
Best Validation Accuracy: 0.8414


In [ ]:

augmentation_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(degrees=30),
    transforms.ToTensor(),
])


train_dataset = CustomDataset(train_df, train_dir, transform=augmentation_transforms)



train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)

In [ ]:
set_seed(5)
device = "cuda"

model = CNN(num_classes=10)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    epochs=30,
    save_path=base_path + "/cnn_agumentation_epoch30.pt"
)

best_val_acc = max(history["val_acc"])

print(
    f"Best Validation Accuracy: "
    f"{best_val_acc:.4f}"
)

Epoch [1/30]: 100%|██████████| 37/37 [00:13<00:00,  2.70it/s, train_acc=0.0727, train_loss=2.3076]


Epoch [1/30] | Train Loss: 2.3076 | Train Acc: 0.0727 | Val Loss: 2.3035 | Val Acc: 0.0966


Epoch [2/30]: 100%|██████████| 37/37 [00:15<00:00,  2.46it/s, train_acc=0.0986, train_loss=2.3041]


Epoch [2/30] | Train Loss: 2.3041 | Train Acc: 0.0986 | Val Loss: 2.3028 | Val Acc: 0.0966


Epoch [3/30]: 100%|██████████| 37/37 [00:14<00:00,  2.60it/s, train_acc=0.0882, train_loss=2.3041]


...

Epoch [28/30] | Train Loss: 0.4637 | Train Acc: 0.8339 | Val Loss: 0.8552 | Val Acc: 0.7172


Epoch [29/30]: 100%|██████████| 37/37 [00:14<00:00,  2.62it/s, train_acc=0.8443, train_loss=0.4370]


Epoch [29/30] | Train Loss: 0.4370 | Train Acc: 0.8443 | Val Loss: 0.7342 | Val Acc: 0.8138


Epoch [30/30]: 100%|██████████| 37/37 [00:14<00:00,  2.53it/s, train_acc=0.8616, train_loss=0.4181]


Epoch [30/30] | Train Loss: 0.4181 | Train Acc: 0.8616 | Val Loss: 0.6745 | Val Acc: 0.8000
Best Validation Accuracy: 0.8138
Best Validation Accuracy: 0.8138
